<div align="center">

# Patra Toolkit: Model Cards & Datasheets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-to-Insight-Center/patra-toolkit/blob/main/examples/notebooks/ModelCardAndDatasheetDemo.ipynb)

</div>

[Patra](https://github.com/Data-to-Insight-Center/patra-knowledge-base) documents AI/ML models and the datasets that trained them as structured, machine-actionable metadata -- **Model Cards** and **Datasheets** -- instead of a README that goes stale. This notebook walks through the full lifecycle of both using the `patra-toolkit` Python client.

**By the end of this notebook, you'll know how to:**
- Build a Model Card and a Datasheet, and submit either to a Patra server
- Discover and retrieve real records from a running Patra server
- *(Optional, advanced)* Stream a live inference run to Patra through CKN

**Prerequisites:** none beyond Python -- the toolkit installs everything else it needs, and this notebook targets a public demo server so you can run it end-to-end without an account.

### Contents
1. [Setup](#1-setup)
2. [Connect to a Patra server](#2-connect)
3. [Model Cards and Datasheets](#3-cards-and-sheets)
4. [Submit to Patra](#4-submit)
5. [Discover Model Cards & Datasheets](#5-discover)
6. [Optional, advanced: stream a live experiment to CKN](#6-experiment)
7. [Next steps](#7-next-steps)

<a id="1-setup"></a>
## 1. Setup

Install the toolkit, then import the two pieces we'll use first: `ModelCard` for documenting a model, and `AIModel` for its performance and framework details. (`Datasheet` too, since we build one right after.)

In [1]:
!pip install -q patra-toolkit

In [2]:
from patra_toolkit import ModelCard, AIModel, Datasheet

<a id="2-connect"></a>
## 2. Connect to a Patra server

Every Model Card and Datasheet is submitted to a **Patra server** -- set its URL once and reuse it for the rest of the notebook. This demo points at a public server, so anonymous submission works out of the box.

Some Patra deployments (e.g. TAPIS-hosted pods) require a JWT for write access. If yours does, leave `tapis_token = None` for now -- [Section 4](#4-submit) covers getting a real one via the client's `authenticate()` method.

In [3]:
patra_server_url = "https://patrabackenddemo.pods.icicleai.tapis.io/"
tapis_token = None  # set via authenticate() in Section 4 if your server requires it

<a id="3-cards-and-sheets"></a>
## 3. Model Cards and Datasheets

A `ModelCard` documents a model -- what it is, who made it, and how it performs -- built field-by-field; only `name` is required, everything else (including an attached `AIModel` for framework, ownership, and metrics) is optional. A `Datasheet` documents the dataset a model was trained on the same way, with `add_*()` convenience methods (`add_title`, `add_creator`, `add_description`, ...) for its DataCite-style fields. Link the two by pointing a model card's `training_datasheet_uuid` at a submitted datasheet's `uuid`.

The full field reference is in the [README](https://github.com/Data-to-Insight-Center/patra-toolkit#building-a-patra-model-card) and [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md) -- [Section 5](#5-discover) below shows what a real submitted record looks like once you look one up.

<a id="4-submit"></a>
## 4. Submit to Patra

Call `validate()` on a `ModelCard` or `Datasheet` to check it against the schema, then `submit(patra_server_url=..., token=...)` -- it re-validates, sends the record to the server, and sets `.uuid` to the id the server assigns.

`submit()` raises `PatraSubmissionError` on a validation or network failure, and `PatraModelExistsError` / `PatraDatasheetExistsError` if an equivalent record already exists on the server -- a Model Card is a duplicate if its `name`, `version`, `author`, and `short_description` all match; a Datasheet, if its title and creator do. Bump `version` to submit a genuinely new one.

Pass `token=<tapis_token>` if your server requires TAPIS authentication (see [Section 2](#2-connect)); get one by calling `.authenticate(username=..., password=...)` on any `ModelCard` instance -- it doesn't touch the instance's data, so any one will do. Either object can also be archived to disk first with `mc.save("model_card.json")` / `ds.save("datasheet.json")`.

There's nothing to run in this section -- the rest of this notebook works against records already on the public demo server, and [Section 5](#5-discover) shows exactly what a submitted record looks like once you look one up.

<a id="5-discover"></a>
## 5. Discover Model Cards & Datasheets

`list_*` returns lightweight summaries, searchable with `q` (a substring match against name/title, author, and short description) and pageable with `skip`/`limit` (server max: 100 per page). `get_*` fetches one full record by `uuid` -- for a Model Card, that includes the nested `ai_model` detail. Both accept `token=` to include private records.

The cells below just pull whatever's already on the public demo server, so they run standalone -- if a list comes back empty, submit a record first (see [Section 4](#4-submit)) and re-run.

### Model Cards

In [4]:
import pandas as pd

model_cards = ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(model_cards)

,id,uuid,name,categories,author,version,short_description,is_gated,is_private,updated_at
0,43,8c517ed0-c9c0-4f57-bb9d-f066ab4ec34e,BioCLIP 2 (via pybioclip),classification,John Bradley / Imageomics Institute,v2.0,Biology foundation model for taxonomic classif...,False,False,2026-05-22T19:50:39.783085+00:00
1,13,b404980f-438a-4760-b358-f6325e6c8f2d,Crop Weed YOLO Model CNW,object detection,Tommy,v1,Custom YOLO model for crop and weed detection.,False,False,2026-05-04T23:18:53.064965+00:00
2,55,b521f10c-6845-44cd-9021-5bf407472633,Deeplab_V3 Engine,Segmentation,Harikesh Byrandurga Gopinath,1.0.0,TensorRT optimized Deeplab_V3 engine for segme...,False,False,2026-06-11T20:41:26.003210+00:00
3,16,31b191a5-121c-4b5f-8266-f99da0f6580f,GoogLeNet for Image Classification,classification,swithana,1.0,Image classification using GoogLeNet.,False,False,2026-05-04T23:18:53.064965+00:00
4,47,62cb5c70-08fc-4899-9522-483274e21ef2,Grounding DINO — grounded open-vocabulary dete...,computer vision,haeparth,1.0,Grounding-oriented detector aligning language ...,False,False,2026-05-05T05:14:47.194483+00:00


Narrow the search with `q`, using a name pulled straight from the results above:

In [5]:
ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, q=model_cards[0]["name"])

[{'id': 43,
  'uuid': '8c517ed0-c9c0-4f57-bb9d-f066ab4ec34e',
  'name': 'BioCLIP 2 (via pybioclip)',
  'categories': 'classification',
  'author': 'John Bradley / Imageomics Institute',
  'version': 'v2.0',
  'short_description': 'Biology foundation model for taxonomic classification and trait prediction.',
  'is_gated': False,
  'is_private': False,
  'updated_at': '2026-05-22T19:50:39.783085+00:00'}]

Fetch that record's full detail, including its `ai_model`:

In [6]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=model_cards[0]["uuid"], token=tapis_token)

{'id': 43,
 'uuid': '8c517ed0-c9c0-4f57-bb9d-f066ab4ec34e',
 'name': 'BioCLIP 2 (via pybioclip)',
 'version': 'v2.0',
 'short_description': 'Biology foundation model for taxonomic classification and trait prediction.',
 'full_description': 'BioCLIP 2 is a large-scale vision-language foundation model for biological understanding. It is trained on the TreeOfLife-200M dataset, which contains 214 million images across 952k taxa. Unlike standard CLIP, it utilizes hierarchical contrastive learning to align image representations with the taxonomic tree of life, allowing it to generalize to unseen species and even predict ecological traits like habitat and life stage.',
 'keywords': 'biology, taxonomy, wildlife, organism detection, zero-shot, pybioclip, tree of life',
 'author': 'John Bradley / Imageomics Institute',
 'input_data': 'https://huggingface.co/datasets/imageomics/TreeOfLife-200M',
 'output_data': 'https://github.com/Imageomics/pybioclip',
 'input_type': 'images',
 'categories': 'cl

### Datasheets

In [7]:
datasheets = Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(datasheets)

,identifier,uuid,title,creator,category,is_private,updated_at
0,14,56688632-9f6d-48da-80d6-76f8a711be19,Big Bird,Swathi V,None,False,2026-07-27T20:59:48.646858+00:00
1,15,d169e2ed-2435-49f0-93ef-207abcc44ede,CKN Inference Demo Images,Demo Author,None,False,2026-07-28T01:09:13.107950+00:00
2,12,9c132373-f663-426d-b5d6-1de5811db3fe,COCO 2017 Train,Swathi V,`object-detection,False,2026-07-27T19:51:14.318935+00:00
3,1,2a7b541d-d3d4-4969-8639-576830ad3d95,Continually Adapt or Not (CAN) Benchmark,ICICLE AI Institute,Camera trap,False,2026-06-03T22:28:37.608303+00:00
4,3,ab55bd2e-5146-4509-b862-cc0292626456,HLO Feature Dataset for Deep Learning Resource...,ICICLE AI Institute,Graph machine learning,False,2026-06-03T22:28:37.608303+00:00


In [8]:
Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, q=datasheets[0]["title"])

[{'identifier': 14,
  'uuid': '56688632-9f6d-48da-80d6-76f8a711be19',
  'title': 'Big Bird',
  'creator': 'Swathi V',
  'category': None,
  'is_private': False,
  'updated_at': '2026-07-27T20:59:48.646858+00:00'}]

In [9]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=datasheets[0]["uuid"], token=tapis_token)

{'identifier': 14,
 'uuid': '56688632-9f6d-48da-80d6-76f8a711be19',
 'publication_year': None,
 'resource_type': 'Dataset',
 'resource_type_general': None,
 'size': None,
 'format': None,
 'version': '1.0.0',
 'is_private': False,
 'updated_at': '2026-07-27T20:59:48.646858+00:00',
 'creators': [{'creator_name': 'Swathi V',
   'name_type': None,
   'lang': None,
   'given_name': None,
   'family_name': None,
   'name_identifier': None,
   'name_identifier_scheme': None,
   'name_id_scheme_uri': None,
   'affiliation': None,
   'affiliation_identifier': None,
   'affiliation_identifier_scheme': None,
   'affiliation_scheme_uri': None}],
 'titles': [{'title': 'Big Bird', 'title_type': None, 'lang': None}],
 'publisher': {'name': 'LILA BC',
  'publisher_identifier': None,
  'publisher_identifier_scheme': None,
  'scheme_uri': None,
  'lang': None},
 'subjects': [],
 'contributors': [],
 'dates': [],
 'alternate_identifiers': [],
 'related_identifiers': [{'related_identifier': 'https://awsc

<a id="6-experiment"></a>
## 6. Optional, advanced: stream a live experiment to CKN

Everything so far submits *static* metadata. `run_experiment()` goes a step further: it downloads a real pretrained model and a batch of images, runs inference, and streams a metric event per image to **CKN** (Kafka), Patra's runtime event-streaming layer -- so results appear in Patra's web app as they're produced.

This section is independent of Sections 1-5 above and needs more than a Python environment, so treat it as optional.

#### Prerequisites
1. A reachable CKN Kafka broker address (e.g. `cknbroker.pods.icicleai.tapis.io:443` -- use whatever your broker's *advertised* external listener actually is, which may not match the port you'd otherwise expect).
2. A `user_id` already registered in the `users` table wherever these events land -- `run_experiment()` does not auto-register users, and there's no safe default.
3. The `experiments` extra: `pip install "patra-toolkit[experiments]"` (installs torch, torchvision, pillow, and confluent-kafka).

`run_experiment()` connects over SSL by default (`use_ssl=True`) -- brokers reached through a TLS-terminating proxy (like a Tapis Pod's external port) need this even if the broker's own internal listener config says `PLAINTEXT`, since that setting only describes the broker's side of the connection *after* the proxy's TLS termination. Pass `use_ssl=False` for a broker with no such proxy in front of it.

If your CKN deployment's Kafka Connect sink connector doesn't require the schema-enveloped JSON format (some do, some accept bare JSON), pass `use_schema_envelope=False` to `run_experiment()`.

In [10]:
!pip install -q "patra-toolkit[experiments]"

### 6.1 A Model Card and Datasheet to run inference against

`run_experiment()` looks up a Model Card and Datasheet by uuid -- rather than building a fresh pair here, this reuses two real records already on the demo server: a Model Card describing a pretrained ResNet50 (with a real, downloadable weights file behind `ai_model.location`), and a Datasheet pointing at the same Lorem Picsum sample images used elsewhere in this notebook.

In [11]:
import torchvision
from patra_toolkit import run_experiment

model_card_uuid = "56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6"
datasheet_uuid = "d169e2ed-2435-49f0-93ef-207abcc44ede"
imagenet_categories = torchvision.models.ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

### 6.2 Run the experiment

`run_experiment()` does the rest on its own: downloads the Model Card and Datasheet by uuid, downloads the model weights and sample images they reference, runs inference, and streams a CKN event per image as it's processed.

`categories` is passed explicitly rather than relying on the fetched Model Card -- `GET /modelcard/{uuid}` doesn't echo back `ai_model.inference_labels`, so it can't be re-derived after the round trip.

In [12]:
result = run_experiment(
    model_card_uuid=model_card_uuid,
    datasheet_uuid=datasheet_uuid,
    patra_server_url=patra_server_url,
    ckn_broker_url="cknbroker.pods.icicleai.tapis.io:443",  # use your CKN broker's advertised external address
    user_id="demo_user",  # replace with your own registered user_id
    token=tapis_token,
    categories=imagenet_categories,
)
result

{'experiment_id': 'experiment-05361ace',
 'model_card_uuid': '56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6',
 'datasheet_uuid': 'd169e2ed-2435-49f0-93ef-207abcc44ede',
 'user_id': 'demo_user',
 'device_id': 'demo-edge-device',
 'domain': 'digital-ag',
 'num_events_produced': 20,
 'events': [{'domain': 'digital-ag',
   'device_id': 'demo-edge-device',
   'experiment_id': 'experiment-05361ace',
   'user_id': 'demo_user',
   'model_id': '56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6',
   'UUID': '3d29cd1f-cf7f-49b2-a4a5-3e8737d77d30',
   'image_name': '0.jpg',
   'ground_truth': None,
   'image_count': 1,
   'image_receiving_timestamp': '2026-07-28T01:17:59Z',
   'image_scoring_timestamp': '2026-07-28T01:17:59Z',
   'image_store_delete_time': '2026-07-28T01:17:59Z',
   'image_decision': 'Save',
   'label': 'notebook',
   'probability': 0.6385801,
   'flattened_scores': '[{"label": "notebook", "probability": 0.6386}, {"label": "laptop", "probability": 0.142}, {"label": "space bar", "probability": 0.0681}, 

### 6.3 View results in the Patra web app

1. **Backend**: `ENABLE_DOMAIN_EXPERIMENTS` needs to be enabled on the Patra server whose database CKN's Kafka Connect sink connector actually writes into -- for this demo, that's `https://patrabackend.pods.icicleai.tapis.io/`, **not** necessarily `patra_server_url` above. The Model Card/Datasheet were fetched from `patra_server_url`, but the streamed events land wherever the connector points, so check results there.
2. **Web app**: set `VITE_SUPPORTS_DOMAIN_EXPERIMENTS=true` in `patra-frontend/app/.env` (it defaults to `false`), then run `npm run dev` from `patra-frontend/app/`.
3. Open the app and click **Digital Agriculture** under Experiments in the sidebar, then select your `user_id` to see this run's summary and per-image results.

You can check the same data directly via the REST API on the server CKN actually writes to:
```bash
curl -s "https://patrabackend.pods.icicleai.tapis.io/experiments/digital-ag/users/demo_user/summary"
```

`result['results_url']` (built from `patra_server_url`) is printed above too, but for a different demo/production database split like this one, it may not show the same data as the curl command above -- it reflects where the Model Card/Datasheet were fetched *from*, not necessarily where CKN's connector writes *to*.

**If nothing shows up**, check your CKN deployment's Kafka Connect logs for errors around the time you ran this -- the sink connector silently drops malformed or unresolvable records (e.g. an unregistered `user_id` or `model_id`, or a schema-envelope mismatch -- see `use_schema_envelope` above) rather than raising anything visible here.

<a id="7-next-steps"></a>
## 7. Next steps

- **Fairness & explainability**: `mc.populate_bias(...)` (via [fairlearn](https://fairlearn.org/)) and `mc.populate_xai(...)` (via [SHAP](https://shap.readthedocs.io/)) can auto-populate bias and feature-importance metrics from a trained model -- see the [README](https://github.com/Data-to-Insight-Center/patra-toolkit#run-fairness-and-explainability-scanners).
- **Schema reference**: every field on `ModelCard` and `AIModel` is documented in [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md).
- **More examples**: framework-specific walkthroughs (PyTorch, TensorFlow, scikit-learn, Hugging Face) live alongside this notebook in [examples/notebooks/](https://github.com/Data-to-Insight-Center/patra-toolkit/tree/main/examples/notebooks).
- **Browse submitted records**: Patra's web app, in this workspace at `patra-frontend/`, lets you search, view, and edit Model Cards and Datasheets from a browser.